In [0]:
# Fonctions et configuration
import re

from pyspark.sql.functions import (
    col,
    current_timestamp,
    lit,
    trim,
    upper,
    when
)

def to_snake_case(column_name):
    name = re.sub(
        r"(.)([A-Z][a-z]+)",
        r"\1_\2",
        column_name
    )
    name = re.sub(
        r"([a-z0-9])([A-Z])",
        r"\1_\2",
        name
    )
    return name.lower()


def normalize_columns(df):
    return df.select([
        col(f"`{column_name}`").alias(
            to_snake_case(column_name)
        )
        for column_name in df.columns
    ])

In [0]:
# nettoyer les tables azure
table_primary_keys = {
    "customers": "customer_id",
    "orders": "order_id",
    "order_lines": "order_line_id",
    "stock_items": "stock_item_id",
    "stock_item_holdings": "stock_item_id",
    "stock_item_transactions": "stock_item_transaction_id",
    "suppliers": "supplier_id",
    "purchase_orders": "purchase_order_id",
    "purchase_order_lines": "purchase_order_line_id"
}

silver_results = []

for table_name, primary_key in table_primary_keys.items():

    bronze_table = f"retail_dev.bronze.{table_name}"
    silver_table = f"retail_dev.silver.{table_name}"

    bronze_df = spark.table(bronze_table)
    normalized_df = normalize_columns(bronze_df)

    silver_df = (
        normalized_df
        .filter(col(primary_key).isNotNull())
        .dropDuplicates([primary_key])
        .withColumn(
            "_silver_processed_at",
            current_timestamp()
        )
    )

    (
        silver_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(silver_table)
    )

    silver_count = spark.table(silver_table).count()

    silver_results.append(
        (silver_table, silver_count, "SUCCESS")
    )

    print(
        f"{silver_table} : "
        f"{silver_count} lignes nettoyées"
    )

In [0]:
# Qualité des événements de stock
bronze_events = (
    spark.table("retail_dev.bronze.stock_events")
    .withColumn("event_type", upper(trim(col("event_type"))))
    .dropDuplicates(["event_id"])
)

valid_products = (
    spark.table("retail_dev.silver.stock_items")
    .select("stock_item_id")
    .distinct()
    .withColumn("_valid_product", lit(True))
)

checked_events = (
    bronze_events
    .join(
        valid_products,
        on="stock_item_id",
        how="left"
    )
    .withColumn(
        "_quarantine_reason",
        when(
            col("event_id").isNull(),
            "MISSING_EVENT_ID"
        )
        .when(
            col("_valid_product").isNull(),
            "UNKNOWN_STOCK_ITEM"
        )
        .when(
            ~col("store_id").between(1, 10),
            "INVALID_STORE_ID"
        )
        .when(
            ~col("event_type").isin(
                "SALE",
                "RESTOCK",
                "RETURN",
                "ADJUSTMENT"
            ),
            "INVALID_EVENT_TYPE"
        )
        .when(
            col("quantity_change") == 0,
            "ZERO_QUANTITY"
        )
        .when(
            (col("event_type") == "SALE")
            & (col("quantity_change") > 0),
            "INVALID_SALE_QUANTITY"
        )
        .when(
            col("event_type").isin("RESTOCK", "RETURN")
            & (col("quantity_change") < 0),
            "INVALID_POSITIVE_EVENT_QUANTITY"
        )
        .when(
            col("event_timestamp").isNull(),
            "MISSING_EVENT_TIMESTAMP"
        )
    )
)

valid_events = (
    checked_events
    .filter(col("_quarantine_reason").isNull())
    .drop("_valid_product", "_quarantine_reason")
    .withColumn(
        "_silver_processed_at",
        current_timestamp()
    )
)

invalid_events = (
    checked_events
    .filter(col("_quarantine_reason").isNotNull())
    .drop("_valid_product")
    .withColumn(
        "_quarantined_at",
        current_timestamp()
    )
)

In [0]:
# Écrire Silver et la quarantaine
(
    valid_events.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("retail_dev.silver.stock_events")
)

quarantine_path = (
    "/Volumes/retail_dev/ops/"
    "quarantine_volume/stock_events"
)

(
    invalid_events.write
    .format("json")
    .mode("overwrite")
    .save(quarantine_path)
)

valid_count = valid_events.count()
invalid_count = invalid_events.count()

print(f"Événements Silver valides : {valid_count}")
print(f"Événements en quarantaine : {invalid_count}")

In [0]:
results_df = spark.createDataFrame(
    silver_results,
    ["silver_table", "row_count", "status"]
)

display(results_df)

display(
    spark.table("retail_dev.silver.stock_events")
    .orderBy(col("event_timestamp").desc())
    .limit(20)
)